# Mini Compiler — Implementasi Fase-Fase Kompilator

**Mata Kuliah:** Teknik Kompilasi  
**Topik:** Lexer, Parser (EBNF), AST, dan TAC  
**Fitur:** Dukungan operator pangkat (`^`) dengan prioritas tertinggi

### Hierarki Operator (terendah → tertinggi)
| Level | Fungsi | Operator |
|-------|--------|----------|
| Rendah | `expr()` | `+` `-` |
| Sedang | `term()` | `*` `/` |
| Tinggi | `power()` | `^` (right-associative) |
| Tertinggi | `factor()` | NUMBER \| IDENT \| `(expr)` |

### EBNF Grammar
```
expr    ::= term   ( ('+' | '-') term   )*
term    ::= power  ( ('*' | '/')  power  )*
power   ::= factor ( '^'          factor )*   ← right-assoc ditangani secara rekursif
factor  ::= NUMBER | IDENT | '(' expr ')'
```

## 1. Definisi Node AST

In [ ]:
import re

class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left  = left
        self.op    = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

class UnaryOp(AST):
    """Node tambahan untuk unary minus, contoh: -x atau -2"""
    def __init__(self, op, operand):
        self.op      = op
        self.operand = operand

class ParserError(Exception):
    pass

## 2. Implementasi Mini Compiler

In [ ]:
class MiniCompiler:
    def __init__(self, source: str, env: dict):
        # TUGAS 1: Perbarui regex agar mengenali simbol '^'
        # Perubahan: tambahkan '\^' ke dalam karakter set operator
        # Sebelum : r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*()\-]'
        # Sesudah : r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*()\-\^]'
        self._tokens = iter(
            re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()|\-\^]', source)
            + ['?']   # sentinel EOF
        )
        self._current     = None
        self._env         = env
        self._temp_count  = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected: str) -> str:
        if self._current != expected and not (
            expected == "ID" and self._current is not None and self._current.isalnum()
        ):
            raise ParserError(f"Expected '{expected}', found '{self._current}'")
        token = self._current
        self.advance()
        return token

    def factor(self):
        """
        factor ::= NUMBER | IDENT | '(' expr ')' | '-' factor
        Tingkat terendah (paling kuat mengikat): literal & grouping.
        """
        token = self._current

        if token == '-':
            self.advance()
            operand = self.factor()
            return UnaryOp('-', operand)

        if token is not None and re.fullmatch(r'\d+(?:\.\d+)?', token):
            self.advance()
            return Num(float(token) if '.' in token else int(token))

        if token and re.fullmatch(r'[a-zA-Z_]\w*', token):
            if token not in self._env:
                raise ParserError(
                    f"Semantic Error: Variabel '{token}' tidak terdefinisi "
                    f"dalam symbol table."
                )
            self.advance()
            return Var(token)

        if token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node

        raise ParserError(f"Token tidak terduga: '{token}'")

    def power(self):
        """
        power ::= factor ( '^' factor )*
        Operator '^' bersifat RIGHT-ASSOCIATIVE:
            2 ^ 3 ^ 2  dibaca sebagai  2 ^ (3 ^ 2) = 512
        """
        node = self.factor()
        bases = [node]
        while self._current == '^':
            self.advance()
            bases.append(self.factor())

        # Bangun pohon dari KANAN ke KIRI (right-associative)
        result = bases[-1]
        for base in reversed(bases[:-1]):
            result = BinOp(left=base, op='^', right=result)
        return result

    def term(self):
        """
        term ::= power ( ('*' | '/') power )*
        Perubahan: self.factor() diganti self.power()
        """
        node = self.power()
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.power())
        return node

    def expr(self):
        """
        expr ::= term ( ('+' | '-') term )*
        Prioritas terendah — dipanggil pertama kali dari luar.
        """
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    def print_ast(self, node, prefix: str = "", is_left: bool = True) -> str:
        connector = "\u251c\u2500\u2500 " if is_left else "\u2514\u2500\u2500 "
        extension = "\u2502   " if is_left else "    "
        if isinstance(node, Num):
            return prefix + connector + str(node.value) + "\n"
        if isinstance(node, Var):
            return prefix + connector + node.name + "\n"
        if isinstance(node, UnaryOp):
            result  = prefix + connector + f"[unary {node.op}]\n"
            result += self.print_ast(node.operand, prefix + extension, False)
            return result
        if isinstance(node, BinOp):
            result  = prefix + connector + f"[{node.op}]\n"
            result += self.print_ast(node.left,  prefix + extension, True)
            result += self.print_ast(node.right, prefix + extension, False)
            return result
        return ""

    def generate_tac(self, node) -> str:
        """
        Menghasilkan instruksi TAC secara rekursif (post-order traversal).
        """
        if isinstance(node, Num):
            return str(node.value)
        if isinstance(node, Var):
            return node.name
        if isinstance(node, UnaryOp):
            val = self.generate_tac(node.operand)
            self._temp_count += 1
            temp = f"t{self._temp_count}"
            print(f"  {temp} = {node.op}{val}")
            return temp
        left_val  = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)
        self._temp_count += 1
        temp = f"t{self._temp_count}"
        print(f"  {temp} = {left_val} {node.op} {right_val}")
        return temp

    def evaluate(self, node):
        if isinstance(node, Num):
            return node.value
        if isinstance(node, Var):
            return self._env[node.name]
        if isinstance(node, UnaryOp):
            if node.op == '-':
                return -self.evaluate(node.operand)
        if isinstance(node, BinOp):
            L = self.evaluate(node.left)
            R = self.evaluate(node.right)
            if node.op == '+': return L + R
            if node.op == '-': return L - R
            if node.op == '*': return L * R
            if node.op == '/':
                if R == 0:
                    raise ZeroDivisionError("Pembagian dengan nol")
                return L / R
            if node.op == '^': return L ** R
        raise ValueError(f"Node tidak dikenal: {type(node)}")

## 3. Uji Coba

In [ ]:
def run_test(source_code: str, symbol_table: dict, label: str = ""):
    separator = "=" * 55
    print(f"\n{separator}")
    if label:
        print(f"  TEST: {label}")
    print(f"  Input       : {source_code}")
    print(f"  Symbol Table: {symbol_table}")
    print(separator)
    try:
        compiler  = MiniCompiler(source_code, symbol_table)
        ast_root  = compiler.expr()

        print("\n  Abstract Syntax Tree:")
        tree_str = compiler.print_ast(ast_root, "  ")
        print("  \u2514\u2500\u2500 " + tree_str.lstrip("  \u251c\u2500\u2500 ").lstrip("  \u2514\u2500\u2500 ").rstrip())

        print("\n  Three Address Code (TAC):")
        compiler._temp_count = 0
        result_temp = compiler.generate_tac(ast_root)
        print(f"  result = {result_temp}")

        try:
            nilai = compiler.evaluate(ast_root)
            print(f"\n  Evaluasi Numerik: {nilai}")
        except Exception:
            print("\n  (Evaluasi numerik tidak tersedia \u2014 ada variabel simbolik)")

    except (ParserError, ZeroDivisionError) as e:
        print(f"\n  ERROR: {e}")


# --- Uji coba wajib dari soal ---
run_test("a ^ 2 + b * c", {'a': 5, 'b': 10, 'c': 2}, "Dari soal: a ^ 2 + b * c")

# --- Uji coba right-associativity ---
run_test("2 ^ 3 ^ 2", {'x': 1}, "Right-assoc: 2^3^2 harus = 2^(3^2) = 512")

# --- Prioritas ^ lebih tinggi dari * ---
run_test("2 + 3 * 4 ^ 2", {}, "Prioritas: 2 + 3 * 4^2 = 2 + 3*16 = 50")

# --- Kurung mengubah prioritas ---
run_test("(2 + 3) ^ 2", {}, "Grouping: (2+3)^2 = 25")

# --- Unary minus + pangkat ---
run_test("x ^ 2 + y ^ 2", {'x': 3, 'y': 4}, "Pythagoras: x^2 + y^2")

# --- Uji semantic error ---
run_test("a + z", {'a': 1}, "Semantic Error: z tidak terdefinisi")

## 4. Jawaban Pertanyaan Refleksi

### Q1. Mengapa `power()` harus dipanggil di dalam `term()`, bukan sebaliknya? Kaitannya dengan Operator Precedence?

Dalam Recursive Descent Parser, **hierarki panggilan fungsi menentukan prioritas operator**. Aturannya:

> Fungsi yang dipanggil **lebih dalam** = operator **lebih kuat**.

Urutan panggilan yang benar:
```
expr()          ← menangani + dan -  (prioritas rendah)
  └─ term()     ← menangani * dan /  (prioritas sedang)
       └─ power()  ← menangani ^    (prioritas tinggi)
            └─ factor()  ← literal, kurung, variabel
```

Ketika `term()` memproses `"3 * 2^4"`:
- Sisi kiri  : `power()` → `factor()` → `Num(3)`
- Operator   : `*`
- Sisi kanan : `power()` → menemukan `'^'` → `BinOp(2, ^, 4)`
- Hasilnya   : `BinOp(Num(3), *, BinOp(Num(2), ^, Num(4)))` = 3 * (2^4) = 48 ✓

Jika `power()` memanggil `term()` (terbalik), maka `'*'` akan diproses lebih dalam dan mendapat prioritas lebih tinggi, menghasilkan `(3*2)^4 = 1296` ✗ — **SALAH**.

### Q2. Apa yang terjadi pada Analisis Semantik jika variabel `z` digunakan namun tidak ada di `symbol_table`?

- **Lexer (fase 1)** TIDAK mendeteksi error ini karena `'z'` adalah token `IDENT` yang valid secara leksikal.
- **Parser (fase 2)** pun tidak mendeteksinya karena `'z'` memenuhi aturan grammar: `factor ::= IDENT`.

Error baru terdeteksi pada fase **Analisis Semantik (fase 3)**, tepatnya saat fungsi `factor()` memeriksa symbol table:

```python
if token not in self._env:
    raise ParserError(
        f"Semantic Error: Variabel '{token}' tidak terdefinisi"
    )
```

Ini disebut **Undeclared Variable Error** — salah satu pemeriksaan semantik paling dasar. Pemrosesan dihentikan sebelum TAC dibangkitkan, mencegah kode yang tidak valid dieksekusi.

### Q3. Mengapa dalam TAC, instruksi untuk `a^2` harus muncul SEBELUM instruksi untuk `+`?

TAC dibangkitkan dengan **post-order traversal** pada AST:
1. Kunjungi subpohon **kiri** → hasilkan TAC-nya
2. Kunjungi subpohon **kanan** → hasilkan TAC-nya
3. Baru hasilkan instruksi untuk **operator saat ini**

Untuk `"a ^ 2 + b * c"`, AST-nya adalah:
```
         [+]
        /   \
     [^]     [*]
    /   \   /   \
   a    2  b     c
```

Post-order traversal menghasilkan urutan:
```
t1 = a ^ 2      ← subpohon kiri [^] diselesaikan dulu
t2 = b * c      ← subpohon kanan [*] diselesaikan
t3 = t1 + t2    ← operator [+] baru dieksekusi
```

Instruksi TAC harus berurutan demikian karena `t1` dan `t2` **belum ada nilainya** saat `t3` dibutuhkan. Inilah prinsip **Dependency Order** dalam code generation: setiap instruksi hanya boleh menggunakan temporari yang sudah didefinisikan pada instruksi sebelumnya.